In [1]:
import xarray as xr
import json
import os, sys
import json

# from src.netcdf import mat_to_xarray
sys.path.append('/Users/yugao/UOP/ORS-processing/src')
from netcdf import read_mat_file, read_metadata_json, save_json, mat_to_xarray, save_to_netcdf

In [2]:
# New depth parameters here (replace example values with actual values)
depth_parameters = {
    'water_depth_from_mooring_diagram': 100,  # Replace with actual value
    'water_depth_from_mooring_log': 105,      # Replace with actual value
    'instrument_depth_from_mooring_diagram': 20,  # Replace with actual value
    'instrument_depth_from_mooring_log': 22,      # Replace with actual value
    'instrument_height_above_bottom': 5       # Replace with actual value
}

## open the external NTAS deep T/S file

In [3]:
ds = xr.open_dataset('/Users/yugao/UOP/ORS-processing/data/external/OS_NTAS_2016_D_TS.nc')

## Export metadata to a JSON file

In [4]:
# Your updated metadata dictionary construction, using ds.sizes instead of ds.dims
metadata = {
    'dimensions': dict(ds.sizes),  # Use ds.sizes here
    'coordinates': {name: {'data': coord.values.tolist(), 'attrs': dict(coord.attrs)} for name, coord in ds.coords.items()},
    'variables': {name: {'dims': var.dims, 'data_type': str(var.dtype), 'attrs': dict(var.attrs)} for name, var in ds.data_vars.items()},
    'attributes': dict(ds.attrs),
    # Assume depth_parameters is defined elsewhere or included here
}

In [5]:
# Update metadata dictionary with depth parameters
metadata = {
    'dimensions': dict(ds.sizes),  # Updated based on FutureWarning advice
    'coordinates': {name: {'data': coord.values.tolist(), 'attrs': dict(coord.attrs)} for name, coord in ds.coords.items()},
    'variables': {name: {'dims': var.dims, 'data_type': str(var.dtype), 'attrs': dict(var.attrs)} for name, var in ds.data_vars.items()},
    'attributes': dict(ds.attrs),
    'depth_parameters': depth_parameters  # Adding depth parameters to metadata
}

# Export updated metadata to a JSON file
with open('../../data/processed/OS_NTAS_2016_D_TS_with_depth_paramter.json', 'w') as f_json:
    json.dump(metadata, f_json, indent=4)

## Meta data

In [6]:
original_json_path = '../../data/metadata/OS_NTAS_2016_D_TS.json'
new_json_path = '../../data/metadata/stratus.json'

## function 1: Extracting Basic Information

In [7]:
def extract_basic_info(mat_data):
    """
    Extract basic information like latitude, longitude, etc.
    """
    basic_info = {
        'latitude': mat_data.get('latitude', None),
        'longitude': mat_data.get('longitude', None),
        'depth': mat_data.get('depth', None)
    }
    return basic_info


In [8]:
mat_filepath = '../../data/external/stratus12_sbe16_1876.mat'
mat_data = read_mat_file(mat_filepath)
extract_basic_info(mat_data)

{'latitude': -19.93844, 'longitude': -85.29266333333334, 'depth': 4502}

## Function 2: process dimensions

In [9]:
def process_dimensions(mat_data):
    """
    Calculate dimensions for the dataset.
    """
    dimensions = {
        'TIME': len(mat_data['mday']),  # Assuming mday represents time and is a 1D array
        'DEPTH': 1,  # Assuming a single depth for simplicity; adjust if your data varies
        'LATITUDE': 1,  # Fixed value for the dataset
        'LONGITUDE': 1  # Fixed value for the dataset
    }
    return dimensions

In [10]:
process_dimensions(mat_data)

{'TIME': 31872, 'DEPTH': 1, 'LATITUDE': 1, 'LONGITUDE': 1}

## Function 3: prcess variable metadata

In [11]:
def process_variables(mat_data):
    '''prcess variable metadata'''
    variables_metadata = {
        'TEMP': {
            'dims': ('TIME', 'DEPTH'),
            'data_type': 'float64',
            'attrs': {
                'long_name': 'Sea water temperature',
                'units': 'degree_Celsius',
                'reference_scale': 'ITS-90',
                'valid_min': min(mat_data['temp']),
                'valid_max': max(mat_data['temp']),
                'QC_indicator': 1.0,
                'QC_procedure': 6.0,
                'instrument': 'SBE-16',
                'comment': 'In-situ measurement, quality controlled'
            }
        },
        'CNDC': {
            'dims': ('TIME', 'DEPTH'),
            'data_type': 'float64',
            'attrs': {
                'long_name': 'Sea water electrical conductivity',
                'units': 'S m-1',
                'valid_min': min(mat_data['cond']),
                'valid_max': max(mat_data['cond']),
                'QC_indicator': 1.0,
                'instrument': 'SBE-16',
                'comment': 'In-situ measurement, quality controlled'
            }
        },
        'PSAL': {
            'dims': ('TIME', 'DEPTH'),
            'data_type': 'float64',
            'attrs': {
                'long_name': 'Sea water practical salinity',
                'units': '1',
                'valid_min': min(mat_data['sal']),
                'valid_max': max(mat_data['sal']),
                'QC_indicator': 1.0,
                'instrument': 'SBE-16',
                'comment': 'Computed from conductivity, quality controlled'
            }
        },
        'sal_sbe': {
            'dims': ('TIME', 'DEPTH'),
            'data_type': 'float64',
            'attrs': {
                'long_name': 'Sea water salinity measured by SBE',
                'units': '1',
                'valid_min': min(mat_data['sal_sbe']),
                'valid_max': max(mat_data['sal_sbe']),
                'QC_indicator': 1.0,
                'instrument': 'SBE-16',
                'comment': 'Direct measurement by SBE, quality controlled'
            }
        }
    }

    # Adjust the above metadata as necessary for accuracy and completeness

    return variables_metadata


In [12]:
process_variables(mat_data)

{'TEMP': {'dims': ('TIME', 'DEPTH'),
  'data_type': 'float64',
  'attrs': {'long_name': 'Sea water temperature',
   'units': 'degree_Celsius',
   'reference_scale': 'ITS-90',
   'valid_min': 1.7675,
   'valid_max': 24.9417,
   'QC_indicator': 1.0,
   'QC_procedure': 6.0,
   'instrument': 'SBE-16',
   'comment': 'In-situ measurement, quality controlled'}},
 'CNDC': {'dims': ('TIME', 'DEPTH'),
  'data_type': 'float64',
  'attrs': {'long_name': 'Sea water electrical conductivity',
   'units': 'S m-1',
   'valid_min': -7.8e-05,
   'valid_max': 5.170328,
   'QC_indicator': 1.0,
   'instrument': 'SBE-16',
   'comment': 'In-situ measurement, quality controlled'}},
 'PSAL': {'dims': ('TIME', 'DEPTH'),
  'data_type': 'float64',
  'attrs': {'long_name': 'Sea water practical salinity',
   'units': '1',
   'valid_min': 0.001377706020971474,
   'valid_max': 34.69117173760024,
   'QC_indicator': 1.0,
   'instrument': 'SBE-16',
   'comment': 'Computed from conductivity, quality controlled'}},
 'sal_s

## Function 4: attributes

In [13]:
from datetime import datetime, timedelta

def process_attributes_direct(mat_data):
# Convert MATLAB datenum to Python datetime
    def matlab_datenum_to_datetime(matlab_datenum):
        python_datetime = datetime.fromordinal(int(matlab_datenum)) + timedelta(days=matlab_datenum%1) - timedelta(days=366)
        return python_datetime

    # Direct extraction of latitude, longitude, and time (mday) values
    latitude = mat_data['latitude']
    longitude = mat_data['longitude']
    mday = mat_data['mday']

    # Convert the first and last MATLAB datenum to Python datetime
    start_datetime = matlab_datenum_to_datetime(mday[0])
    end_datetime = matlab_datenum_to_datetime(mday[-1])

    # Format start and end datetime to ISO format
    start_date_iso = start_datetime.strftime('%Y-%m-%dT%H:%M:%SZ')
    end_date_iso = end_datetime.strftime('%Y-%m-%dT%H:%M:%SZ')

    # Calculate duration in days (approximation)
    duration_days = (end_datetime - start_datetime).days
    duration_iso = f'P{duration_days}D'

    # Prepare the attributes dictionary
    attributes = {
        'geospatial_lat_min': latitude,
        'geospatial_lat_max': latitude,
        'geospatial_lon_min': longitude,
        'geospatial_lon_max': longitude,
        'geospatial_lat_units': 'degrees_north',
        'geospatial_lon_units': 'degrees_east',
        'geospatial_vertical_min': 0.78,  # Assuming this is constant; adjust as necessary
        'geospatial_vertical_max': 70.0,  # Assuming this is constant; adjust as necessary
        'geospatial_vertical_units': 'meters',
        'geospatial_vertical_positive': 'down',
        'time_coverage_start': start_date_iso,
        'time_coverage_end': end_date_iso,
        'time_coverage_duration': duration_iso,
        'time_coverage_resolution': 'PT1H',  # Assuming hourly resolution; adjust as necessary
    }

    return attributes

In [14]:
process_attributes_direct(mat_data)

{'geospatial_lat_min': -19.93844,
 'geospatial_lat_max': -19.93844,
 'geospatial_lon_min': -85.29266333333334,
 'geospatial_lon_max': -85.29266333333334,
 'geospatial_lat_units': 'degrees_north',
 'geospatial_lon_units': 'degrees_east',
 'geospatial_vertical_min': 0.78,
 'geospatial_vertical_max': 70.0,
 'geospatial_vertical_units': 'meters',
 'geospatial_vertical_positive': 'down',
 'time_coverage_start': '2012-05-14T00:59:59Z',
 'time_coverage_end': '2014-03-09T00:29:59Z',
 'time_coverage_duration': 'P663D',
 'time_coverage_resolution': 'PT1H'}

In [15]:
def create_json(mat_data):
    """
    Combines dimensions, variables, and attributes into a final JSON structure.

    Args:
        mat_data: The MATLAB data loaded as a dictionary.

    Returns:
        A dictionary representing the final JSON structure.
    """
    # Process dimensions
    dimensions = process_dimensions(mat_data)
    
    # Process variables
    variables = process_variables(mat_data)
    
    # Process attributes
    attributes = process_attributes_direct(mat_data)

    # Combine everything into a final structure
    final_structure = {
        'dimensions': dimensions,
        'variables': variables,
        'attributes': attributes
    }
    
    return final_structure



In [16]:
# Now you can generate the final structure for your dataset
final_json = create_json(mat_data)

In [17]:
final_json

{'dimensions': {'TIME': 31872, 'DEPTH': 1, 'LATITUDE': 1, 'LONGITUDE': 1},
 'variables': {'TEMP': {'dims': ('TIME', 'DEPTH'),
   'data_type': 'float64',
   'attrs': {'long_name': 'Sea water temperature',
    'units': 'degree_Celsius',
    'reference_scale': 'ITS-90',
    'valid_min': 1.7675,
    'valid_max': 24.9417,
    'QC_indicator': 1.0,
    'QC_procedure': 6.0,
    'instrument': 'SBE-16',
    'comment': 'In-situ measurement, quality controlled'}},
  'CNDC': {'dims': ('TIME', 'DEPTH'),
   'data_type': 'float64',
   'attrs': {'long_name': 'Sea water electrical conductivity',
    'units': 'S m-1',
    'valid_min': -7.8e-05,
    'valid_max': 5.170328,
    'QC_indicator': 1.0,
    'instrument': 'SBE-16',
    'comment': 'In-situ measurement, quality controlled'}},
  'PSAL': {'dims': ('TIME', 'DEPTH'),
   'data_type': 'float64',
   'attrs': {'long_name': 'Sea water practical salinity',
    'units': '1',
    'valid_min': 0.001377706020971474,
    'valid_max': 34.69117173760024,
    'QC_in